# Praktikum Pemfilteran Citra Digital (FilterLab)

Notebook ini adalah lembar kerja terpandu untuk media pembelajaran **FilterLab**. Ada enam praktikum yang harus dijalankan **berurutan dari atas ke bawah**, karena praktikum berikutnya memakai variabel dari praktikum sebelumnya.

**Cara memakai**
1. Pilih *File > Save a copy in Drive* agar salinanmu bisa diedit.
2. Jalankan tiap sel dengan Shift+Enter.
3. Ubah nilai pada baris yang diberi komentar `# UBAH`, jalankan ulang, lalu amati perubahannya.
4. Tulis jawaban di sel teks tepat di bawah tiap pertanyaan analisis.

**Identitas** (isi di sel berikut)

In [ ]:
NAMA = ""  # UBAH: isi namamu
NIM = ""   # UBAH: isi NIM-mu
print("Nama:", NAMA, "| NIM:", NIM)

## Praktikum 1: Memuat gambar

**Tujuan:** Membaca citra dengan OpenCV, mengonversi warna, dan mengetahui ukuran serta tipe datanya.

**Dasar teori singkat:** OpenCV membaca citra sebagai array NumPy dengan urutan kanal BGR. Matplotlib mengharapkan RGB, sehingga perlu dikonversi sebelum ditampilkan.

**Langkah:**
1. Jalankan sel dan unggah satu gambar (jpg atau png).
2. Catat ukuran (tinggi, lebar, kanal) yang dicetak.
3. Amati citra asli yang ditampilkan.

**Parameter yang boleh diubah:** `MAKS` (ukuran sisi terpanjang setelah diperkecil, misalnya 256, 512).

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()  # pilih satu gambar
nama_berkas = list(uploaded.keys())[0]
image = cv2.imread(nama_berkas)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Perkecil jika terlalu besar agar proses cepat
MAKS = 512  # UBAH
h, w = image.shape[:2]
if max(h, w) > MAKS:
    skala = MAKS / max(h, w)
    image = cv2.resize(image, (int(w * skala), int(h * skala)))

print("Ukuran (tinggi, lebar, kanal):", image.shape)
print("Tipe data:", image.dtype)
plt.imshow(image); plt.title("Citra asli"); plt.axis("off"); plt.show()

### Pertanyaan analisis 1
Berapa jumlah piksel gambarmu (tinggi × lebar), dan berapa kanal warnanya?

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Praktikum 2: Menambah noise

**Tujuan:** Membuat dua versi citra bernoise (Gaussian dan salt-and-pepper) sebagai bahan uji filter.

**Dasar teori singkat:** Noise Gaussian ditambahkan dari distribusi normal dengan simpangan baku sigma. Salt-and-pepper mengubah sebagian piksel secara acak menjadi 0 (pepper) atau 255 (salt).

**Langkah:**
1. Jalankan sel dan lihat ketiga citra.
2. Naikkan `sigma` dan `p`, jalankan ulang, dan amati perubahannya.
3. Catat nilai di mana noise mulai mengganggu.

**Parameter yang boleh diubah:** `sigma` (misalnya 10, 25, 50) dan `p` (misalnya 0.02, 0.05, 0.10).

In [ ]:
np.random.seed(0)  # agar hasil bisa diulang

def gaussian_noise(img, sigma=25):
    n = np.random.normal(0, sigma, img.shape)
    return np.clip(img + n, 0, 255).astype(np.uint8)

def salt_pepper(img, p=0.05):
    out = img.copy()
    m = np.random.rand(*img.shape[:2])
    out[m < p / 2] = 0
    out[m > 1 - p / 2] = 255
    return out

def tampil(gambar, judul, ukuran=(16, 4)):
    fig, ax = plt.subplots(1, len(gambar), figsize=ukuran)
    for a, im, t in zip(ax, gambar, judul):
        a.imshow(im); a.set_title(t); a.axis("off")
    plt.show()

sigma = 25  # UBAH
p = 0.05    # UBAH
noisy_g = gaussian_noise(image, sigma)
noisy_sp = salt_pepper(image, p)
tampil([image, noisy_g, noisy_sp], ["Asli", f"Gaussian (sigma={sigma})", f"Salt-and-pepper (p={p})"], (15, 5))

### Pertanyaan analisis 2
Pada nilai sigma dan p berapa noise mulai mengganggu pengenalan objek di gambarmu?

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Praktikum 3: Mean Filter

**Tujuan:** Menerapkan Mean Filter dan melihat pengaruh ukuran kernel.

**Dasar teori singkat:** Mean Filter merata-ratakan piksel dalam jendela k × k. Fungsi `cv2.blur` melakukannya dengan kernel berbobot sama.

**Langkah:**
1. Jalankan dengan k = 3.
2. Ulangi dengan k = 5 dan k = 7.
3. Perhatikan bintik putih dan hitam pada hasil salt-and-pepper.

**Parameter yang boleh diubah:** Ukuran kernel `k` (bilangan ganjil).

In [ ]:
k = 3  # UBAH: 3, 5, atau 7
mean_sp = cv2.blur(noisy_sp, (k, k))
mean_g = cv2.blur(noisy_g, (k, k))
tampil([noisy_sp, mean_sp, noisy_g, mean_g],
       ["Salt-and-pepper", f"Mean k={k} pada SP", "Gaussian", f"Mean k={k} pada Gaussian"])

### Pertanyaan analisis 3
Apakah bintik salt-and-pepper hilang setelah Mean Filter? Jelaskan alasannya.

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Praktikum 4: Median Filter

**Tujuan:** Menerapkan Median Filter dan membandingkannya dengan Mean pada salt-and-pepper.

**Dasar teori singkat:** Median mengambil nilai tengah dari k × k piksel yang sudah diurutkan, sehingga nilai ekstrem terbuang. Ukuran kernel `cv2.medianBlur` harus ganjil dan lebih dari 1.

**Langkah:**
1. Gunakan nilai `k` yang sama dengan Praktikum 3.
2. Bandingkan hasil Median dan Mean pada citra salt-and-pepper.
3. Perhatikan kejernihan tepi objek.

**Parameter yang boleh diubah:** Ukuran kernel `k` (ganjil).

In [ ]:
median_sp = cv2.medianBlur(noisy_sp, k)
median_g = cv2.medianBlur(noisy_g, k)
tampil([noisy_sp, median_sp, noisy_g, median_g],
       ["Salt-and-pepper", f"Median k={k} pada SP", "Gaussian", f"Median k={k} pada Gaussian"])

### Pertanyaan analisis 4
Mengapa Median jauh lebih bersih pada citra salt-and-pepper?

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Praktikum 5: Gaussian Filter

**Tujuan:** Menerapkan Gaussian Filter dan memahami peran parameter sigma.

**Dasar teori singkat:** Gaussian memberi bobot yang menurun dari pusat. Jika sigma bernilai 0, OpenCV menghitungnya otomatis dari ukuran kernel.

**Langkah:**
1. Jalankan dengan sigma otomatis (0).
2. Coba beberapa nilai sigma pada eksperimen kedua.
3. Bandingkan dengan hasil Mean pada citra Gaussian.

**Parameter yang boleh diubah:** Ukuran kernel `k` dan daftar `SIGMA`.

In [ ]:
gauss_g = cv2.GaussianBlur(noisy_g, (k, k), 0)
gauss_sp = cv2.GaussianBlur(noisy_sp, (k, k), 0)
tampil([noisy_g, gauss_g, mean_g, noisy_sp, gauss_sp],
       ["Gaussian", f"Gaussian filter k={k}", f"Mean k={k}", "Salt-and-pepper", f"Gaussian filter k={k} pada SP"], (20, 4))

# Eksperimen sigma
SIGMA = [0.5, 1.5, 3]  # UBAH
hasil_sigma = [cv2.GaussianBlur(noisy_g, (k, k), s) for s in SIGMA]
tampil(hasil_sigma, [f"sigma={s}" for s in SIGMA], (15, 5))

### Pertanyaan analisis 5
Apa bedanya hasil Gaussian Filter dan Mean Filter pada citra bernoise Gaussian? Apa pengaruh sigma?

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Praktikum 6: Kernel dan perbandingan kuantitatif

**Tujuan:** Membandingkan ketiga filter secara objektif dengan PSNR pada beberapa ukuran kernel.

**Dasar teori singkat:** PSNR membandingkan hasil dengan citra asli: makin tinggi makin dekat ke asli. Bandingkan hanya pada jenis noise dan ukuran kernel yang sama.

**Langkah:**
1. Jalankan sel dan amati tabel serta grafik yang muncul.
2. Tentukan filter terbaik untuk tiap jenis noise.
3. Cari ukuran kernel dengan PSNR tertinggi.

**Parameter yang boleh diubah:** `KERNEL` (daftar ukuran ganjil), serta `sigma` dan `p` pada Praktikum 2 untuk mengubah tingkat noise.

In [ ]:
def f_mean(x, k): return cv2.blur(x, (k, k))
def f_median(x, k): return cv2.medianBlur(x, k)
def f_gauss(x, k): return cv2.GaussianBlur(x, (k, k), 0)
FILTER = {"Mean": f_mean, "Median": f_median, "Gaussian": f_gauss}

KERNEL = (3, 5, 7)  # UBAH
baris = []
for noise, noisy in [("Gaussian", noisy_g), ("Salt-and-pepper", noisy_sp)]:
    baris.append({"Noise": noise, "Filter": "(tanpa filter)", "Kernel": 0,
                  "PSNR (dB)": round(cv2.PSNR(image, noisy), 2)})
    for kk in KERNEL:
        for nama, fn in FILTER.items():
            baris.append({"Noise": noise, "Filter": nama, "Kernel": kk,
                          "PSNR (dB)": round(cv2.PSNR(image, fn(noisy, kk)), 2)})

df = pd.DataFrame(baris)
display(df)

tabel = df[df["Filter"] != "(tanpa filter)"].pivot_table(
    index=["Noise", "Kernel"], columns="Filter", values="PSNR (dB)")
display(tabel)
print("Filter terbaik per kombinasi noise dan kernel:")
print(tabel.idxmax(axis=1))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, noise in zip(ax, ["Gaussian", "Salt-and-pepper"]):
    sub = df[(df["Noise"] == noise) & (df["Filter"] != "(tanpa filter)")]
    for nama in FILTER:
        s = sub[sub["Filter"] == nama]
        a.plot(s["Kernel"], s["PSNR (dB)"], marker="o", label=nama)
    a.set_title(f"Noise {noise}"); a.set_xlabel("Ukuran kernel"); a.set_ylabel("PSNR (dB)")
    a.set_xticks(list(KERNEL)); a.legend(); a.grid(alpha=0.3)
plt.show()

### Pertanyaan analisis 6
Untuk tiap jenis noise, filter dan ukuran kernel mana yang terbaik? Apakah kernel terbesar selalu menang? Gunakan angka dari tabel sebagai bukti.

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Kesimpulan akhir

Tuliskan kesimpulan 3 sampai 4 kalimat tentang cara memilih filter berdasarkan jenis noise, dengan merujuk pada hasil praktikummu.

**Jawaban:** *(klik dua kali sel ini, lalu tulis jawabanmu di sini)*

## Simpan hasil (opsional)

Jalankan sel berikut untuk mengunduh tabel PSNR sebagai CSV. Sertakan berkas ini bersama jawabanmu bila diminta pengajar.

In [ ]:
df.to_csv("hasil_psnr.csv", index=False)
files.download("hasil_psnr.csv")